In [1]:
# + Khởi động môi trường và import các hàm cần dùng
import os
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve().parent
sys.path.append(str(PROJECT_ROOT / "src"))

from data_processing import (
    load_all_source_tables,
    clean_source_tables,
    build_daily_model_base,
    build_monthly_panel,
    infer_model_schema,
)

In [2]:
# + Đọc toàn bộ fact và dimension từ BigQuery
source = load_all_source_tables(
    project_id="dwm-final-487216",
    dataset_id="nong_san_dbscl",
)

source = clean_source_tables(source)

for name, df in source.items():
    print(f"{name}: {df.shape}")

fact: (4944, 17)
dim_date: (2191, 6)
dim_economy: (3, 4)
dim_fertilizer_type: (3, 4)
dim_region: (1, 4)
dim_rice_type: (5, 4)
dim_weather: (4, 5)


In [3]:
# + Kiểm tra đúng bản chất của fact trước khi dựng panel
fact_df = source["fact"].copy()

print("Số dòng có rice_id:", fact_df["rice_id"].notna().sum())
print("Số dòng có fert_id:", fact_df["fert_id"].notna().sum())

display(fact_df.head(20))

Số dòng có rice_id: 3090
Số dòng có fert_id: 1854


,date_id,region_id,eco_id,weather_id,rice_id,fert_id,price_rice,price_fert,avg_temperature,max_temp,min_temp,rainfall_mm,avg_wind_speed,production_ton,area_harvest_ha,yield_kg_per_ha,cpi_change_value
0,2024-01-02,1,1,1,<NA>,F01,NaN,15500.0,27.5,29.0,26.0,0.5,7.8,20330000.0,2954000.0,6882.193636,0.31
1,2024-01-02,1,1,1,<NA>,F02,NaN,12900.0,27.5,29.0,26.0,0.5,7.8,20330000.0,2954000.0,6882.193636,0.31
2,2024-01-02,1,1,1,<NA>,F03,NaN,10950.0,27.5,29.0,26.0,0.5,7.8,20330000.0,2954000.0,6882.193636,0.31
3,2024-01-02,1,1,1,R01,<NA>,9000.0,NaN,27.5,29.0,26.0,0.5,7.8,20330000.0,2954000.0,6882.193636,0.31
4,2024-01-02,1,1,1,R02,<NA>,9600.0,NaN,27.5,29.0,26.0,0.5,7.8,20330000.0,2954000.0,6882.193636,0.31
5,2024-01-02,1,1,1,R03,<NA>,9300.0,NaN,27.5,29.0,26.0,0.5,7.8,20330000.0,2954000.0,6882.193636,0.31
6,2024-01-02,1,1,1,R04,<NA>,5451.0,NaN,27.5,29.0,26.0,0.5,7.8,20330000.0,2954000.0,6882.193636,0.31
7,2024-01-02,1,1,1,R05,<NA>,6600.0,NaN,27.5,29.0,26.0,0.5,7.8,20330000.0,2954000.0,6882.193636,0.31
8,2024-01-04,1,1,1,<NA>,F01,NaN,15500.0,27.2,28.7,25.7,0.3,11.6,20330000.0,2954000.0,6882.193636,0.31
9,2024-01-04,1,1,1,<NA>,F02,NaN,14400.0,27.2,28.7,25.7,0.3,11.6,20330000.0,2954000.0,6882.193636,0.31


In [4]:
# + Dựng daily base đúng logic join giữa dòng gạo và dòng phân
daily_base = build_daily_model_base(source)

print("Daily base shape:", daily_base.shape)
display(daily_base.head(20))
print(daily_base.columns.tolist())

Daily base shape: (1854, 33)


,date_id,region_id,rice_id,price_rice,eco_id,weather_id,avg_temperature,max_temp,min_temp,rainfall_mm,...,region_name,rice_name,rice_group,unit,cpi_status,market_signal,economy_unit,weather_type,severity_level,agriculture_impact
0,2024-01-02,1,R01,9000.0,1,1,27.5,29.0,26.0,0.5,...,Đồng bằng sông Cửu Long,Gạo nguyên liệu IR 504,Lúa tươi,VND/kg,Tăng,Cảnh báo lạm phát,%,Ôn hòa,Bình thường,Lý tưởng cho lúa sinh trưởng
1,2024-01-04,1,R01,7400.0,1,1,27.2,28.7,25.7,0.3,...,Đồng bằng sông Cửu Long,Gạo nguyên liệu IR 504,Lúa tươi,VND/kg,Tăng,Cảnh báo lạm phát,%,Ôn hòa,Bình thường,Lý tưởng cho lúa sinh trưởng
2,2024-01-05,1,R01,7400.0,1,1,27.0,28.5,25.5,0.4,...,Đồng bằng sông Cửu Long,Gạo nguyên liệu IR 504,Lúa tươi,VND/kg,Tăng,Cảnh báo lạm phát,%,Ôn hòa,Bình thường,Lý tưởng cho lúa sinh trưởng
3,2024-01-07,1,R01,7000.0,1,1,27.3,28.8,25.8,0.0,...,Đồng bằng sông Cửu Long,Gạo nguyên liệu IR 504,Lúa tươi,VND/kg,Tăng,Cảnh báo lạm phát,%,Ôn hòa,Bình thường,Lý tưởng cho lúa sinh trưởng
4,2024-01-08,1,R01,6900.0,1,1,27.3,28.8,25.8,0.6,...,Đồng bằng sông Cửu Long,Gạo nguyên liệu IR 504,Lúa tươi,VND/kg,Tăng,Cảnh báo lạm phát,%,Ôn hòa,Bình thường,Lý tưởng cho lúa sinh trưởng
5,2024-01-09,1,R01,7900.0,1,1,27.2,28.7,25.7,1.6,...,Đồng bằng sông Cửu Long,Gạo nguyên liệu IR 504,Lúa tươi,VND/kg,Tăng,Cảnh báo lạm phát,%,Ôn hòa,Bình thường,Lý tưởng cho lúa sinh trưởng
6,2024-01-10,1,R01,9700.0,1,1,27.5,29.0,26.0,0.5,...,Đồng bằng sông Cửu Long,Gạo nguyên liệu IR 504,Lúa tươi,VND/kg,Tăng,Cảnh báo lạm phát,%,Ôn hòa,Bình thường,Lý tưởng cho lúa sinh trưởng
7,2024-01-11,1,R01,6700.0,1,1,27.4,28.9,25.9,0.3,...,Đồng bằng sông Cửu Long,Gạo nguyên liệu IR 504,Lúa tươi,VND/kg,Tăng,Cảnh báo lạm phát,%,Ôn hòa,Bình thường,Lý tưởng cho lúa sinh trưởng
8,2024-01-12,1,R01,7800.0,1,1,26.8,28.3,25.3,0.1,...,Đồng bằng sông Cửu Long,Gạo nguyên liệu IR 504,Lúa tươi,VND/kg,Tăng,Cảnh báo lạm phát,%,Ôn hòa,Bình thường,Lý tưởng cho lúa sinh trưởng
9,2024-01-14,1,R01,6600.0,1,1,26.0,27.5,24.5,0.0,...,Đồng bằng sông Cửu Long,Gạo nguyên liệu IR 504,Lúa tươi,VND/kg,Tăng,Cảnh báo lạm phát,%,Ôn hòa,Bình thường,Lý tưởng cho lúa sinh trưởng


['date_id', 'region_id', 'rice_id', 'price_rice', 'eco_id', 'weather_id', 'avg_temperature', 'max_temp', 'min_temp', 'rainfall_mm', 'avg_wind_speed', 'production_ton', 'area_harvest_ha', 'yield_kg_per_ha', 'cpi_change_value', 'price_fert_F01', 'price_fert_F02', 'price_fert_F03', 'year', 'quarter', 'month', 'day', 'crop_season', 'region_name', 'rice_name', 'rice_group', 'unit', 'cpi_status', 'market_signal', 'economy_unit', 'weather_type', 'severity_level', 'agriculture_impact']


In [5]:
# + Kiểm tra chắc chắn rằng giá phân đã được pivot thành cột rộng
fert_cols = [c for c in daily_base.columns if str(c).startswith("price_fert_")]
print("Fertilizer columns:", fert_cols)

display(daily_base[fert_cols].head(10))
display(daily_base[fert_cols].notna().sum().sort_values(ascending=False))

Fertilizer columns: ['price_fert_F01', 'price_fert_F02', 'price_fert_F03']


,price_fert_F01,price_fert_F02,price_fert_F03
0,15500.0,12900.0,10950.0
1,15500.0,14400.0,14400.0
2,14800.0,11200.0,10450.0
3,14700.0,10900.0,9950.0
4,14700.0,11900.0,10125.0
5,NaN,NaN,NaN
6,NaN,NaN,NaN
7,NaN,NaN,NaN
8,NaN,NaN,NaN
9,NaN,NaN,NaN


price_fert_F01    1104
price_fert_F02    1104
price_fert_F03    1104
dtype: int64

In [6]:
# + Gom về panel tháng để chuẩn bị cho mô hình 1-2-3 tháng tới
panel = build_monthly_panel(
    daily_base,
    horizons=(1, 2, 3),
    max_target_lag=6,
    max_feature_lag=3,
    rolling_windows=(2, 3, 6),
)

print("Monthly panel shape:", panel.shape)
display(panel.head(20))

Monthly panel shape: (81, 163)


,date_id,region_id,rice_id,price_rice,avg_temperature,max_temp,min_temp,rainfall_mm,avg_wind_speed,production_ton,...,price_fert_F03_roll_std_6,price_rice_diff_1,price_rice_pct_change_1,target_t_plus_1,direction_t_plus_1,target_t_plus_2,direction_t_plus_2,target_t_plus_3,direction_t_plus_3,row_is_latest
0,2024-01-01,1,R01,7781.818182,27.081818,28.581818,25.581818,0.390909,10.681818,20330000.0,...,NaN,NaN,NaN,7371.428571,0.0,7879.629630,1.0,7533.333333,0.0,False
1,2024-02-01,1,R01,7371.428571,27.964286,29.464286,26.464286,0.078571,13.407143,20330000.0,...,435.067089,-410.389610,-0.052737,7879.629630,1.0,7533.333333,1.0,7920.833333,1.0,False
2,2024-03-01,1,R01,7879.629630,29.518519,31.018519,28.018519,0.274074,14.781481,8200000.0,...,403.237892,508.201058,0.068942,7533.333333,0.0,7920.833333,1.0,7122.550000,0.0,False
3,2024-04-01,1,R01,7533.333333,31.312500,32.812500,29.812500,0.150000,15.241667,11160000.0,...,568.135640,-346.296296,-0.043948,7920.833333,1.0,7122.550000,0.0,7170.833333,0.0,False
4,2024-05-01,1,R01,7920.833333,28.816667,30.316667,27.316667,9.825000,14.041667,11160000.0,...,540.769271,387.500000,0.051438,7122.550000,0.0,7170.833333,0.0,7596.153846,0.0,False
5,2024-06-01,1,R01,7122.550000,27.845000,29.345000,26.345000,10.410000,14.475000,11160000.0,...,487.500796,-798.283333,-0.100783,7170.833333,1.0,7596.153846,1.0,7228.571429,1.0,False
6,2024-07-01,1,R01,7170.833333,27.125000,28.625000,25.625000,8.620833,17.541667,11160000.0,...,409.668870,48.283333,0.006779,7596.153846,1.0,7228.571429,1.0,6942.307692,0.0,False
7,2024-08-01,1,R01,7596.153846,27.815385,29.315385,26.315385,7.957692,16.150000,8200000.0,...,434.842411,425.320513,0.059313,7228.571429,0.0,6942.307692,0.0,7936.956522,1.0,False
8,2024-09-01,1,R01,7228.571429,27.400000,28.900000,25.900000,8.466667,19.885714,8200000.0,...,642.529139,-367.582418,-0.048391,6942.307692,0.0,7936.956522,1.0,7658.333333,1.0,False
9,2024-10-01,1,R01,6942.307692,26.603846,28.103846,25.103846,15.026923,13.157692,8200000.0,...,505.598958,-286.263736,-0.039602,7936.956522,1.0,7658.333333,1.0,6295.000000,0.0,False


In [7]:
# + Kiểm tra schema cuối cùng dùng cho mô hình
schema = infer_model_schema(panel)

print("Date col:", schema.date_col)
print("Target col:", schema.target_col)
print("Group cols:", schema.group_cols)
print("Numeric feature count:", len(schema.numeric_feature_cols))
print("Categorical feature count:", len(schema.categorical_feature_cols))

print("\nMột vài numeric features:")
print(schema.numeric_feature_cols[:20])

print("\nMột vài categorical features:")
print(schema.categorical_feature_cols[:20])

Date col: date_id
Target col: price_rice
Group cols: ['region_id', 'rice_id']
Numeric feature count: 143
Categorical feature count: 9

Một vài numeric features:
['avg_temperature', 'max_temp', 'min_temp', 'rainfall_mm', 'avg_wind_speed', 'production_ton', 'area_harvest_ha', 'yield_kg_per_ha', 'cpi_change_value', 'price_fert_F01', 'price_fert_F02', 'price_fert_F03', 'eco_id', 'weather_id', 'quarter', 'month', 'year', 'month_num', 'year_num', 'month_sin']

Một vài categorical features:
['region_name', 'rice_name', 'rice_group', 'crop_season', 'cpi_status', 'market_signal', 'weather_type', 'severity_level', 'agriculture_impact']


In [8]:
# + Nhìn nhanh tương quan giữa giá gạo và các biến chính
candidate_cols = [
    "price_rice",
    "avg_temperature",
    "max_temp",
    "min_temp",
    "rainfall_mm",
    "avg_wind_speed",
    "production_ton",
    "area_harvest_ha",
    "yield_kg_per_ha",
    "cpi_change_value",
]
candidate_cols += [c for c in panel.columns if str(c).startswith("price_fert_")]
candidate_cols += [c for c in panel.columns if str(c).startswith("price_rice_lag_")]

candidate_cols = [c for c in candidate_cols if c in panel.columns]
corr_df = panel[candidate_cols].corr(numeric_only=True)

target_corr = (
    corr_df["price_rice"]
    .drop("price_rice", errors="ignore")
    .sort_values(key=lambda s: s.abs(), ascending=False)
    .to_frame("corr_with_price_rice")
)

display(target_corr.head(30))

,corr_with_price_rice
price_fert_F01_roll_mean_3,0.793500
price_fert_F01_roll_mean_6,0.790025
price_fert_F01_roll_mean_2,0.776378
price_fert_F02_roll_mean_2,0.768130
price_fert_F02_roll_mean_3,0.763998
price_fert_F02_roll_mean_6,0.762688
price_fert_F01_lag_1,0.749606
price_rice_lag_1,0.741314
price_fert_F02,0.734531
price_fert_F02_lag_1,0.733765


In [9]:
group_size = (
    panel.groupby(["region_id", "rice_id"])["date_id"]
    .nunique()
    .reset_index(name="n_months")
)
display(group_size.sort_values("n_months"))

,region_id,rice_id,n_months
0,1,R01,27
1,1,R02,27
2,1,R03,27
